gam

In [ ]:
# train_gam_model.py
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pygam import LinearGAM, s, f
import joblib

df = pd.read_csv("./data/train_dataset.csv")
y = df["추정일간클릭수"]
X = df[["1개당가격", "최근4주클릭수_비율", "weekday", "pum_name"]]
X = pd.get_dummies(X, columns=["pum_name", "weekday"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gam = LinearGAM(s(0) + s(1) + f(2) + f(3) + f(4)).fit(X_train, y_train)

y_pred = gam.predict(X_test)
print("테스트셋 R²:", r2_score(y_test, y_pred))
print("테스트셋 MSE:", mean_squared_error(y_test, y_pred))

# 모델과 컬럼 구조 함께 저장
gam_package = {
    "model": gam,
    "columns": list(X.columns)  # 학습에 사용한 컬럼 구조 그대로 저장
}
joblib.dump(gam_package, "./models/gam_model.pkl")


nlp

In [ ]:
# train_mlp_model.py
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

# 데이터 로드
df = pd.read_csv("./data/train_dataset.csv")
y = df["추정일간클릭수"].values
X = df[["1개당가격", "최근4주클릭수_비율", "weekday", "pum_name"]]
X = pd.get_dummies(X, columns=["pum_name", "weekday"], drop_first=True)

# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 텐서 변환
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)   # 첫 은닉층만 사용
        self.fc2 = nn.Linear(32, 1)           # 바로 출력층
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MLPRegressor(input_dim=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 학습 루프
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# 평가
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).numpy()

print("테스트셋 R²:", r2_score(y_test, y_pred))
print("테스트셋 MSE:", mean_squared_error(y_test, y_pred))

# 모델과 스케일러 저장
package = {
    "model_state": model.state_dict(),
    "scaler": scaler,
    "input_dim": X_train.shape[1],
    "columns": list(X.columns)
}
joblib.dump(package, "./models/nn_model.pkl")